# eJagruti — House Price Prediction with Simple Linear Regression

## Complete Step-by-Step Machine Learning Tutorial

This notebook follows the complete workflow covered in **Images 1–5**:

1. Problem Statement
2. Raw Dataset
3. Understand the Dataset
4. Identify Missing Values
5. Identify Invalid Values
6. Detect Outliers using IQR
7. Clean the Dataset
8. Exploratory Data Analysis (EDA)
9. Prepare X and Y
10. Train/Test Split
11. Understand the Linear Regression Formula
12. Build the Model in Python
13. Get the Regression Equation
14. Predict a New House Price
15. Calculate Residuals
16. Evaluate the Model
17. Examine the Regression Line
18. Check Model Assumptions
19. Final Model & Prediction

> **Important data correction:** The original infographic sequence contained a couple of illustrative counting inconsistencies. With the 20-row dataset below, `Area_sqft` has **1 missing value (5%)**, not 2. Following the stated cleaning rules, the mathematically consistent cleaned dataset has **18 rows after removing the invalid value, missing target, and IQR outlier**. This notebook calculates every result directly rather than hard-coding infographic numbers.

## 1. Problem Statement

We want to build a **Simple Linear Regression** model that predicts the selling price of a house from its area in square feet.

- **Independent variable (X):** `Area_sqft`
- **Dependent variable (Y):** `Price_lakh`
- **Goal:** Predict `Price_lakh` from `Area_sqft`

The model has the form:

$$
Price = b_0 + b_1(Area)
$$

Where:

- $b_0$ = intercept
- $b_1$ = slope

## 2. Raw Dataset

The dataset intentionally contains realistic data-quality problems:

- a missing `Area_sqft`
- a missing `Price_lakh`
- a negative area value
- an extreme house-area/price observation

We will **not clean the data immediately**. First, we inspect it step by step.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 20-row raw dataset
data = [
    ("H01",  850,    42),
    ("H02", 1000,    48),
    ("H03", 1100,    52),
    ("H04", 1200,    58),
    ("H05", 1350,    63),
    ("H06", 1500,    70),
    ("H07", 1600,    75),
    ("H08", 1750,    82),
    ("H09", 1900,    88),
    ("H10", 2000,    95),
    ("H11", 2200,   102),
    ("H12", 2400,   112),
    ("H13", -500,    30),   # invalid: negative area
    ("H14", 1250,   np.nan), # missing target
    ("H15", np.nan,  65),   # missing feature
    ("H16", 1450,    68),
    ("H17", 1550,    73),
    ("H18",12000,   900),   # IQR outlier
    ("H19", 1800,    85),
    ("H20", 2100,    97),
]

df = pd.DataFrame(data, columns=["ID", "Area_sqft", "Price_lakh"])

df

## 3. Understand the Dataset

Before changing anything, inspect:

- number of rows and columns
- data types
- descriptive statistics
- the first few rows

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nDescriptive statistics:")
display(df.describe())

## 4. Identify Missing Values

We check both the **count** and **percentage** of missing values.

### Handling strategy

- Missing `Area_sqft`: use **median imputation**
- Missing `Price_lakh`: remove the row because the target value is unavailable

For supervised regression, a row without the target cannot be used to train the model.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Percentage": (df.isna().mean() * 100).round(2)
})

display(missing_summary)

## 5. Identify Invalid Values

A house cannot have a negative physical area.

Therefore:

`Area_sqft < 0` → invalid observation

We identify invalid records before performing the outlier analysis.

In [ ]:
invalid_area = df[df["Area_sqft"] < 0]

print("Invalid rows:")
display(invalid_area)

## 6. Detect Outliers Using the IQR Method

We use the **Interquartile Range (IQR)** method.

### Formula

$$
IQR = Q_3 - Q_1
$$

$$
Lower\ Bound = Q_1 - 1.5(IQR)
$$

$$
Upper\ Bound = Q_3 + 1.5(IQR)
$$

We first exclude the impossible negative area, because an invalid value should not determine the legitimate-data outlier boundaries.

An outlier is **not automatically an error**. We inspect it and decide whether it should be retained or removed.

In [ ]:
# Remove the logically invalid negative area only for the purpose of
# calculating legitimate Area_sqft quartiles.
area_for_iqr = df.loc[df["Area_sqft"] >= 0, "Area_sqft"].dropna()

Q1 = area_for_iqr.quantile(0.25)
Q3 = area_for_iqr.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"Lower Bound = {lower_bound:.2f}")
print(f"Upper Bound = {upper_bound:.2f}")

outliers = df[
    (df["Area_sqft"] < lower_bound) |
    (df["Area_sqft"] > upper_bound)
]

print("\nPotential Area_sqft outliers:")
display(outliers)

## 7. Clean the Dataset

We now apply the decisions from Steps 4–6.

### Cleaning sequence

1. Remove the invalid negative-area row.
2. Remove rows where `Price_lakh` is missing.
3. Fill missing `Area_sqft` with the median of the valid observations.
4. Recalculate the IQR boundaries on the resulting usable data.
5. Remove the extreme IQR outlier (`H18`).

This gives us a consistent modeling dataset.

In [ ]:
clean_df = df.copy()

# 1. Remove invalid negative area
clean_df = clean_df[clean_df["Area_sqft"] >= 0].copy()

# 2. Remove rows with missing target
clean_df = clean_df.dropna(subset=["Price_lakh"]).copy()

# 3. Median-impute missing Area_sqft
area_median = clean_df["Area_sqft"].median()
clean_df["Area_sqft"] = clean_df["Area_sqft"].fillna(area_median)

# 4. Recalculate IQR after the invalid value is removed and the
#    feature is complete.
Q1_clean = clean_df["Area_sqft"].quantile(0.25)
Q3_clean = clean_df["Area_sqft"].quantile(0.75)
IQR_clean = Q3_clean - Q1_clean
lower_clean = Q1_clean - 1.5 * IQR_clean
upper_clean = Q3_clean + 1.5 * IQR_clean

# 5. Remove IQR outliers
clean_df = clean_df[
    clean_df["Area_sqft"].between(lower_clean, upper_clean)
].copy()

clean_df = clean_df.reset_index(drop=True)

print("Original rows:", len(df))
print("Clean rows:", len(clean_df))
print("\nFinal cleaned dataset:")
display(clean_df)

## 8. Exploratory Data Analysis (EDA)

Now that the data is clean, visualize the relationship between:

- X = `Area_sqft`
- Y = `Price_lakh`

We expect a generally positive relationship: larger houses tend to have higher prices in this example.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(clean_df["Area_sqft"], clean_df["Price_lakh"])
plt.xlabel("Area (sq ft)")
plt.ylabel("Price (₹ lakh)")
plt.title("Area vs Price — Cleaned Dataset")
plt.grid(alpha=0.25)
plt.show()

print("Correlation coefficient:")
print(clean_df["Area_sqft"].corr(clean_df["Price_lakh"]))

## 9. Prepare X and Y

Machine-learning libraries usually expect:

- `X` as the feature matrix
- `y` as the target vector

Because this is **simple** linear regression, we use one feature:

`Area_sqft`

and one target:

`Price_lakh`.

In [ ]:
X = clean_df[["Area_sqft"]]
y = clean_df["Price_lakh"]

print("X:")
display(X.head())

print("y:")
display(y.head())

print("X shape:", X.shape)
print("y shape:", y.shape)

## 10. Train/Test Split

We keep some data aside to test the model on observations it did not train on.

Here we use:

- **80% training data**
- **20% testing data**
- `random_state=42` for reproducibility

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 11. Understand the Linear Regression Formula

Simple Linear Regression fits a straight line:

$$
\hat{Y} = b_0 + b_1X
$$

### Slope

$$
b_1 =
\frac{
\sum (X_i-\bar X)(Y_i-\bar Y)
}{
\sum (X_i-\bar X)^2
}
$$

### Intercept

$$
b_0 = \bar Y - b_1\bar X
$$

The slope tells us how much the predicted price changes for one additional square foot.

### Manual calculation of slope and intercept

The next cell calculates the coefficients from the **training data** manually, so the formula is not just theoretical.

In [ ]:
x_train_values = X_train["Area_sqft"].to_numpy()
y_train_values = y_train.to_numpy()

x_bar = x_train_values.mean()
y_bar = y_train_values.mean()

b1_manual = (
    np.sum((x_train_values - x_bar) * (y_train_values - y_bar))
    / np.sum((x_train_values - x_bar) ** 2)
)

b0_manual = y_bar - b1_manual * x_bar

print(f"Training X mean (x̄) = {x_bar:.4f}")
print(f"Training Y mean (ȳ) = {y_bar:.4f}")
print(f"Manual slope (b1) = {b1_manual:.6f}")
print(f"Manual intercept (b0) = {b0_manual:.6f}")

## 12. Build the Model in Python

Now we use scikit-learn's `LinearRegression`.

The model learns the intercept and slope from the training data.

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully.")

## 13. Get the Regression Equation

The trained model exposes:

- `model.intercept_` → $b_0$
- `model.coef_[0]` → $b_1$

We use them to write the final regression equation.

In [ ]:
b0 = model.intercept_
b1 = model.coef_[0]

print(f"Intercept (b0): {b0:.6f}")
print(f"Slope (b1):     {b1:.6f}")

print(f"\nRegression Equation:")
print(f"Price = {b0:.4f} + {b1:.6f} × Area_sqft")

## 14. Predict a New House Price

Suppose a new house has:

**Area = 2,000 sq ft**

The model predicts:

$$
Price = b_0 + b_1(2000)
$$

In [ ]:
new_house = pd.DataFrame({"Area_sqft": [2000]})

predicted_price = model.predict(new_house)[0]

print(f"Predicted price for a 2,000 sq ft house = ₹{predicted_price:.2f} lakh")

## 15. Calculate Residuals

A residual tells us the prediction error for an observation.

$$
Residual = Actual - Predicted
$$

- Positive residual → model predicted too low.
- Negative residual → model predicted too high.
- Residual close to zero → prediction is close to actual.

We calculate residuals for the **test set**, because these observations were not used to train the model.

In [ ]:
test_predictions = model.predict(X_test)

residuals_df = pd.DataFrame({
    "ID": clean_df.loc[X_test.index, "ID"],
    "Area_sqft": X_test["Area_sqft"].values,
    "Actual_Price_lakh": y_test.values,
    "Predicted_Price_lakh": test_predictions,
    "Residual_lakh": y_test.values - test_predictions
})

residuals_df = residuals_df.sort_values("ID").reset_index(drop=True)

display(residuals_df)

## 16. Evaluate the Model

We use four common regression metrics.

### MAE — Mean Absolute Error

$$
MAE = \frac{1}{n}\sum |y-\hat y|
$$

Average absolute prediction error.

### MSE — Mean Squared Error

$$
MSE = \frac{1}{n}\sum (y-\hat y)^2
$$

Squares errors, so larger errors receive more weight.

### RMSE — Root Mean Squared Error

$$
RMSE = \sqrt{MSE}
$$

Expressed in the same unit as the target.

### R² — Coefficient of Determination

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Describes how much variation in the target is explained by the model.

> The exact metric values depend on the cleaned dataset and the train/test split. They are calculated below rather than copied from an illustrative infographic.

In [ ]:
mae = mean_absolute_error(y_test, test_predictions)
mse = mean_squared_error(y_test, test_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, test_predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2]
})

display(metrics)

## 17. Examine the Regression Line

The regression line shows the relationship learned by the model.

We plot:

- cleaned observations
- fitted regression line

In [ ]:
x_line = np.linspace(
    clean_df["Area_sqft"].min(),
    clean_df["Area_sqft"].max(),
    200
)

y_line = model.predict(pd.DataFrame({"Area_sqft": x_line}))

plt.figure(figsize=(9, 5))
plt.scatter(
    clean_df["Area_sqft"],
    clean_df["Price_lakh"],
    label="Actual data"
)
plt.plot(
    x_line,
    y_line,
    linestyle="--",
    label="Regression line"
)

plt.xlabel("Area (sq ft)")
plt.ylabel("Price (₹ lakh)")
plt.title("House Area vs Price with Regression Line")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 18. Check Model Assumptions

Simple Linear Regression works best when several assumptions are reasonably satisfied.

### Main checks

1. **Linearity** — relationship between X and Y is approximately linear.
2. **Independence** — observations/errors should be independent.
3. **Homoscedasticity** — residual spread should be reasonably constant.
4. **Normality of residuals** — especially relevant for inference.
5. **Influential observations** — extreme observations should not dominate the model.

These checks are partly statistical and partly visual. A small demonstration dataset cannot establish every assumption with certainty.

In [ ]:
# Residual diagnostics
residuals = y_test - test_predictions
fitted = test_predictions

plt.figure(figsize=(9, 5))
plt.scatter(fitted, residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price (₹ lakh)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Predicted Values")
plt.grid(alpha=0.25)
plt.show()

# Simple residual summary
print("Residual mean:", residuals.mean())
print("Residual standard deviation:", residuals.std(ddof=1))

### Normality visual check

A histogram gives a quick visual indication of the residual distribution.

For a serious project, additional diagnostics such as Q-Q plots, formal tests, leverage, Cook's distance, and domain validation can be added.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(residuals, bins=6, edgecolor="black")
plt.xlabel("Residual (₹ lakh)")
plt.ylabel("Frequency")
plt.title("Distribution of Test Residuals")
plt.grid(alpha=0.25)
plt.show()

## 19. Final Model & Prediction

We now summarize the complete workflow.

### Final equation

$$
Price = b_0 + b_1(Area)
$$

The model can be used to estimate the price of a house from its area, **within the range and assumptions represented by this small training dataset**.

For example, we already predicted the price of a 2,000 sq ft house above.

### Complete ML pipeline

**Raw Data → Inspect → Handle Missing Values → Remove Invalid Values → IQR Outlier Detection → Clean Data → EDA → X/Y → Train/Test Split → Train Model → Equation → Predict → Residuals → Evaluate → Diagnostics → Final Prediction**

In [ ]:
print("=" * 70)
print("eJagruti — FINAL MODEL SUMMARY")
print("=" * 70)

print(f"Original dataset size : {len(df)} rows")
print(f"Clean dataset size    : {len(clean_df)} rows")
print(f"Training rows         : {len(X_train)}")
print(f"Testing rows          : {len(X_test)}")

print(f"\nRegression equation:")
print(f"Price = {b0:.4f} + {b1:.6f} × Area_sqft")

print(f"\nEvaluation:")
print(f"MAE  = {mae:.4f} lakh")
print(f"MSE  = {mse:.4f}")
print(f"RMSE = {rmse:.4f} lakh")
print(f"R²   = {r2:.4f}")

print(f"\nExample:")
print(f"Predicted price for 2,000 sq ft = ₹{predicted_price:.2f} lakh")

print("\nNext step: use a larger real-world dataset and validate the model")
print("with stronger cross-validation, diagnostics, and domain-specific features.")

# Key Learning Points

### Data preparation
- Never train directly on unchecked raw data.
- Missing values, invalid values, and outliers need separate treatment.
- An outlier is not automatically an error.

### Simple Linear Regression
- One feature → one straight-line relationship.
- `b0` is the intercept.
- `b1` is the slope.
- Prediction is `b0 + b1 × X`.

### Model evaluation
- MAE is easy to interpret.
- MSE penalizes large errors more strongly.
- RMSE is in the same unit as the target.
- R² describes explained variation.

### Most important idea

**A model is not just the regression formula.**

A complete machine-learning workflow includes:

**Data Quality → Cleaning → Exploration → Training → Prediction → Error Analysis → Evaluation → Assumption Checks**